# 🔬 Skin Lesion Classification (HAM10000) — EfficientNet + Grad-CAM

Classifies 7 skin-lesion types (incl. **melanoma**) from dermatoscopy. Two things most notebooks get wrong, done right here:

1. **Severe class imbalance** (benign nevi dominate) → inverse-frequency weighted loss + we report **balanced accuracy**, not raw accuracy.
2. **The HAM10000 leakage trap** → we split by **`lesion_id`**, not by image. The dataset's 10,015 images come from only 7,470 physical lesions, so a naive image-level split leaks the same lesion into train *and* test and **inflates the score**. Our number is the honest, leak-free one.

Grad-CAM confirms the model attends to the lesion. GPU-only. *Upvotes appreciated 🙏*

In [ ]:
!pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q grad-cam

## Setup — force GPU + auto-locate data

In [ ]:
import os
from pathlib import Path
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, balanced_accuracy_score

assert torch.cuda.is_available(), "GPU required"
_ = nn.Conv2d(3,4,3).cuda()(torch.randn(1,3,8,8,device="cuda")); torch.cuda.synchronize()
DEVICE = torch.device("cuda"); print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
DX = ["akiec","bcc","bkl","df","mel","nv","vasc"]

## 1. Load metadata & see the imbalance

In [ ]:
def find_meta():
    for base in ["/kaggle/input", os.path.expanduser("~/.cache/kagglehub")]:
        for c in Path(base).rglob("HAM10000_metadata*.csv"): return c
    raise FileNotFoundError("metadata not found")
csv = find_meta(); root = csv.parent
df = pd.read_csv(csv)
paths = {p.stem: str(p) for p in root.rglob("*.jpg")}
df["path"] = df.image_id.map(paths); df = df.dropna(subset=["path"]).reset_index(drop=True)
df["label"] = df.dx.map({d:i for i,d in enumerate(DX)})
print("images:", len(df))
plt.figure(figsize=(8,3)); df.dx.value_counts().plot.bar(color="#c44e52")
plt.title("Severe class imbalance (nv dominates)"); plt.show()

## ⚠️ The HAM10000 leakage trap — and how we avoid it

HAM10000 has **10,015 images but only 7,470 unique lesions** — 1,956 lesions were photographed more than once. If you split randomly *by image* (the default in most notebooks), copies of the same physical lesion land in both train and test. The model then partly *memorizes* lesions rather than *diagnoses* them, and the reported accuracy is optimistically high.

**Fix:** split by `lesion_id` with `GroupShuffleSplit` so no lesion is shared across the split. The accuracy below is lower than the leaky version you'll see elsewhere — but it's the number that actually reflects performance on **new patients**.

In [ ]:
MEAN=torch.tensor([0.485,0.456,0.406]).view(3,1,1); STD=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
class Skin(Dataset):
    def __init__(self, d, train=False): self.d, self.train = d.reset_index(drop=True), train
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r=self.d.iloc[i]; img=cv2.resize(cv2.cvtColor(cv2.imread(r["path"]),cv2.COLOR_BGR2RGB),(224,224))
        if self.train and np.random.rand()<0.5: img=img[:,::-1]
        x=torch.from_numpy(np.ascontiguousarray(img)).permute(2,0,1).float()/255.
        return (x-MEAN)/STD, int(r["label"])

# ⚠️ GROUPED split by lesion_id (avoids the well-known HAM10000 leakage).
# 10,015 images come from only 7,470 unique lesions - 1,956 lesions have several
# photos. A plain image-level split puts the SAME physical lesion in both train and
# test, so the model "recognizes" test lesions it already saw -> inflated accuracy.
# We split by lesion so no lesion is shared across the split.
from sklearn.model_selection import GroupShuffleSplit
gss=GroupShuffleSplit(n_splits=1,test_size=0.2,random_state=42)
tr_idx,te_idx=next(gss.split(df,df.label,groups=df.lesion_id))
tr_df,te_df=df.iloc[tr_idx].reset_index(drop=True),df.iloc[te_idx].reset_index(drop=True)
assert set(tr_df.lesion_id)&set(te_df.lesion_id)==set(), "lesion leaked across split!"
print(f"train imgs={len(tr_df)}  test imgs={len(te_df)}  (no lesion shared across the split)")
dl_tr=DataLoader(Skin(tr_df,True),32,shuffle=True,num_workers=2,pin_memory=True)
dl_te=DataLoader(Skin(te_df),32,num_workers=2,pin_memory=True)

## 2. Train — with a class-weighted loss for the imbalance

In [ ]:
model=timm.create_model("efficientnet_b0",pretrained=True,num_classes=7).to(DEVICE)
freq=tr_df.label.value_counts().sort_index().values
w=torch.tensor(freq.sum()/(7*freq)).float().to(DEVICE)   # inverse-frequency weights for imbalance
crit=nn.CrossEntropyLoss(weight=w); opt=torch.optim.AdamW(model.parameters(),lr=3e-4,weight_decay=1e-5); scaler=torch.cuda.amp.GradScaler()
for ep in range(6):
    model.train(); tot=0
    for x,y in dl_tr:
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad()
        with torch.autocast("cuda"): loss=crit(model(x),y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot+=loss.item()
    print(f"epoch {ep+1}/6 loss={tot/len(dl_tr):.4f}")

## 3. Evaluate — balanced accuracy is the fair metric here

In [ ]:
model.eval(); ys,ps=[],[]
with torch.inference_mode():
    for x,y in dl_te: ps+=model(x.to(DEVICE)).argmax(1).cpu().tolist(); ys+=y.tolist()
acc=np.mean(np.array(ys)==np.array(ps)); bacc=balanced_accuracy_score(ys,ps)
print(f"TEST accuracy={acc:.3f}  balanced_acc={bacc:.3f}")
print(classification_report(ys,ps,target_names=DX))
ConfusionMatrixDisplay(confusion_matrix(ys,ps),display_labels=DX).plot(cmap="Blues",xticks_rotation=45); plt.title(f"bal-acc={bacc:.3f}"); plt.tight_layout(); plt.show()

## 4. 🔍 Grad-CAM explainability

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
layer=[m for m in model.modules() if isinstance(m,nn.Conv2d)][-1]; ds=Skin(te_df.reset_index(drop=True))
fig,axes=plt.subplots(2,4,figsize=(13,7))
with GradCAM(model=model,target_layers=[layer]) as cam:
    for ax,idx in zip(axes.ravel(), np.linspace(0,len(ds)-1,8).astype(int)):
        x,y=ds[idx]
        rgb=np.clip(x.permute(1,2,0).numpy()*np.array([0.229,0.224,0.225])+np.array([0.485,0.456,0.406]),0,1).astype(np.float32)
        ax.imshow(show_cam_on_image(rgb, cam(input_tensor=x[None].to(DEVICE))[0], use_rgb=True))
        ax.set_title(DX[y],fontsize=9); ax.axis('off')
plt.suptitle('Grad-CAM: model should attend to the lesion'); plt.tight_layout(); plt.show()

## Conclusion

A class-weighted EfficientNet handles the HAM10000 imbalance, and Grad-CAM confirms lesion-focused attention. Crucially, the score here comes from a **lesion-grouped split** so it is leak-free and reflects generalization to new patients, making it directly comparable across models (and honestly lower than image-split numbers). Next steps: focal loss, oversampling, patient metadata features, and test-time augmentation. *Thanks: upvotes welcome!*